# E2 — Лестница идентификации суррогата (ядро)

Факторная абляция **denoise × optimizer × library × degree** с метриками одношагового и многошагового прогноза (rollout-RMSE@{4,20,96}, доля расходящихся), разреженности и κ. Два шлюза: **MPC-embeddability** (degree-1 встраивается в do-mpc) и **прозрачность** (знак/размерность + структурная устойчивость по бутстрэпу). Рекомендованный рецепт выбирается по Парето и **замораживается** (`recipe_frozen.json`) — предрегистрация против цикличности (§1.4.1).

In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P

# FAST_MODE smoke (tiny data) vs article-grade. Toggle via env var ARTICLE_FAST=0.
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location)
CORR, PRICES = ECON["corridors"], ECON["prices"]
print("FAST_MODE", FAST_MODE, "| seeds", tuple(pc.seeds), "| budgets", pc.budgets_days,
      "| n_days_train/test", pc.n_days_train, pc.n_days_test)

FAST_MODE True | seeds (0, 1) | budgets (1, 3, 5) | n_days_train/test 5 3


## Загрузка датасетов E1 (или сбор при отсутствии)

In [2]:
def _load_or_collect():
    try:
        parts = [U.load_dataset(RES / "datasets" / f"e1_train_{y}.npz") for y in pc.train_years]
        train = U.aggregate_trajectories(parts, pc.base_cfg(pc.n_days_train))
        test = U.load_dataset(RES / "datasets" / "e1_test_2020.npz")
    except Exception:
        train = U.collect_rule_based_dataset(pc.cfg_for(pc.train_scenarios()[0], seed=0), n_days=pc.n_days_train, prbs_scale=0.3)
        test = U.collect_rule_based_dataset(pc.cfg_for(pc.test_scenario(), seed=1), n_days=pc.n_days_test, prbs_scale=0.0)
    return train, test
train, test = _load_or_collect()
print("train rows", len(train.states), "| test rows", len(test.states))

train rows 960 | test rows 288


## Прогон лестницы идентификации + шлюзы

In [3]:
DENOISE = ["none", "savgol", "kalman"]
OPT = ["stlsq", "sr3", "ensemble", "constrained"]
VARIANTS = [("physics", 1), ("physics_no_cross", 1), ("raw", 1), ("physics", 2)]
if FAST_MODE:
    DENOISE = ["none", "savgol"]; OPT = ["stlsq", "ensemble"]; VARIANTS = [("physics", 1), ("raw", 1), ("physics", 2)]
nboot = 8 if FAST_MODE else 15
rows = []
for (var, deg) in VARIANTS:
    for den in DENOISE:
        for opt in OPT:
            if deg == 2 and opt in ("sr3", "constrained"):
                continue
            b = U.fit_sindy(train, feature_variant=var, library_degree=deg, optimizer=opt,
                            denoise=den, period=float(pc.period),
                            ensemble_models=(8 if FAST_MODE else 20),
                            metadata={"label": f"{var}_d{deg}_{opt}_{den}"})
            ev = U.evaluate_sindy(b, test)
            one = ev[(ev.metric_scope=="one_step") & (ev.state=="t_in")]["rmse"].iloc[0]
            roll = ev[(ev.metric_scope=="rollout") & (ev.state=="t_in")]
            rmax = roll[roll.horizon == roll.horizon.max()]
            div = float(rmax["failed_rollouts"].iloc[0]) / max(1, float(rmax["attempted_rollouts"].iloc[0]))
            nz = int(np.count_nonzero(b.model.coefficients())); tot = b.model.coefficients().size
            emb = U.mpc_embeddability_gate(b, pc.cfg_for(pc.test_scenario()), start_date=pc.test_scenario()["start_date"])
            trg = U.transparency_gate(b, train, n_boot=nboot) if deg == 1 else {"passed": False, "sign_pass_rate": np.nan, "structural_stability": np.nan}
            rows.append({"variant": var, "degree": deg, "optimizer": opt, "denoise": den,
                         "one_step_t_rmse": one, "rollout_t_rmse": float(rmax["rmse"].iloc[0]),
                         "diverged_frac": div, "nonzero": nz, "sparsity": 1 - nz/tot,
                         "kappa": b.condition_number, "embeddable": emb["embeddable"],
                         "mpc_step_ms": emb["mpc_step_ms"], "sign_pass": trg["sign_pass_rate"],
                         "structural_stability": trg["structural_stability"], "transparent": trg["passed"]})
ladder = pd.DataFrame(rows)
U.save_table(ladder, RES / "tables" / "e2_ladder.csv")
ladder.sort_values(["diverged_frac", "rollout_t_rmse"]).head(12)

,variant,degree,optimizer,denoise,one_step_t_rmse,rollout_t_rmse,diverged_frac,nonzero,sparsity,kappa,embeddable,mpc_step_ms,sign_pass,structural_stability,transparent
2,physics,1,stlsq,savgol,3.127915,4.987765,0.0,5,0.924242,36.750354,True,8.7621,1.0,0.750000,True
6,raw,1,stlsq,savgol,3.127915,4.987765,0.0,3,0.933333,7.480497,True,8.0454,0.0,1.000000,False
3,physics,1,ensemble,savgol,3.125767,4.994423,0.0,5,0.924242,36.750354,True,8.7122,1.0,0.850000,True
7,raw,1,ensemble,savgol,3.125415,4.997061,0.0,3,0.933333,7.480497,True,7.8925,0.0,1.000000,False
1,physics,1,ensemble,none,2.100714,5.009842,0.0,37,0.439394,42.354475,True,11.0921,0.5,0.867857,True
10,physics,2,stlsq,savgol,3.205614,5.139526,0.0,11,0.985507,36.750354,False,NaN,NaN,NaN,False
0,physics,1,stlsq,none,1.987817,5.263355,0.0,32,0.515152,42.354475,True,12.3466,0.5,0.906250,True
4,raw,1,stlsq,none,2.308107,5.653805,0.0,16,0.644444,7.040343,True,12.1170,0.0,0.968750,False
5,raw,1,ensemble,none,2.352552,5.677332,0.0,18,0.600000,7.040343,True,12.0062,0.0,0.933824,False
11,physics,2,ensemble,savgol,3.191586,23.567134,0.0,10,0.986825,36.750354,False,NaN,NaN,NaN,False


## Выбор рецепта по Парето и заморозка (предрегистрация)

In [4]:
cand = ladder[(ladder.degree == 1) & (ladder.embeddable) & (ladder.transparent)].copy()
if cand.empty:
    cand = ladder[(ladder.degree == 1) & (ladder.embeddable)].copy()
cand = cand.sort_values(["diverged_frac", "rollout_t_rmse", "nonzero"])
best = cand.iloc[0]
recipe = {"feature_variant": best["variant"], "library_degree": int(best["degree"]),
          "optimizer": best["optimizer"], "denoise": best["denoise"]}
U.save_json(RES / "recipe_frozen.json", {"recipe": recipe,
            "selected_by": "pareto(diverged_frac, rollout_rmse, sparsity); gates=embeddable+transparent",
            "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")})
print("FROZEN RECIPE:", recipe)

FROZEN RECIPE: {'feature_variant': 'physics', 'library_degree': 1, 'optimizer': 'stlsq', 'denoise': 'savgol'}


## Прозрачность рекомендованного рецепта: уравнения и знак-проверки

In [5]:
b = U.fit_sindy(train, feature_variant=recipe["feature_variant"], library_degree=recipe["library_degree"],
                optimizer=recipe["optimizer"], denoise=recipe["denoise"], period=float(pc.period),
                metadata={"label": "recommended"})
coef = U.coefficient_table(b); U.save_table(coef, RES / "tables" / "e2_coefficients.csv")
signs = U.sign_check_table(b); U.save_table(signs, RES / "tables" / "e2_sign_checks.csv")
U.plot_coefficient_heatmap(b, RES / "figures")
signs

,equation,term,expected_sign,coefficient,verdict,interpretation
0,co2,dc_uVent,negative,0.000000,missing_or_zero,ventilation should reduce indoor CO2 gradient
1,rh,h_uVent,negative,-0.063428,consistent,ventilation should reduce humidity
2,t_in,t_uBoil,positive,0.000000,missing_or_zero,heating should increase or maintain temperature
3,t_in,S_eff,positive,0.000000,missing_or_zero,solar gain should increase or maintain tempera...


## Стабильность многошагового прогноза vs бюджет данных (рекомендованный рецепт)

In [6]:
srows = []
for bud in pc.budgets_days:
    sub = train.subset_steps(bud * pc.steps_per_day)
    bb = U.fit_sindy(sub, feature_variant=recipe["feature_variant"], library_degree=recipe["library_degree"],
                     optimizer=recipe["optimizer"], denoise=recipe["denoise"], period=float(pc.period))
    ev = U.evaluate_sindy(bb, test); roll = ev[(ev.metric_scope=="rollout") & (ev.state=="t_in")]
    r = roll[roll.horizon == roll.horizon.max()]
    srows.append({"budget_days": bud, "rollout_t_rmse": float(r["rmse"].iloc[0]),
                  "diverged_frac": float(r["failed_rollouts"].iloc[0]) / max(1, float(r["attempted_rollouts"].iloc[0]))})
sdf = pd.DataFrame(srows); U.save_table(sdf, RES / "tables" / "e2_stability_vs_budget.csv")
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(sdf.budget_days, sdf.rollout_t_rmse, marker="o")
ax.set_xlabel("бюджет данных, сут"); ax.set_ylabel("rollout T RMSE"); ax.set_title("Стабильность прогноза vs бюджет"); ax.grid(alpha=.3)
U.save_figure(fig, RES / "figures" / "e2_stability_vs_budget.png"); plt.close(fig); sdf

,budget_days,rollout_t_rmse,diverged_frac
0,1,914.690875,0.95
1,3,6.303855,0.00
2,5,5.158144,0.00


**Итог E2.** Получена таблица абляции с шлюзами; выбран и заморожен рецепт идентификации (предрегистрация); зафиксированы уравнения и знак-проверки прозрачности.